# Day 4-5: Text Classification with scikit-learn 📚

Welcome to the advanced text classification module! In this notebook, we'll explore various machine learning algorithms for text classification using scikit-learn.

## Learning Objectives
- Understand different text classification algorithms
- Implement and compare multiple ML models
- Learn about model evaluation and hyperparameter tuning
- Handle multi-class classification problems
- Build a production-ready text classifier

## What You'll Learn
1. **Classification Algorithms**: Naive Bayes, SVM, Random Forest, Logistic Regression
2. **Model Evaluation**: Cross-validation, classification metrics
3. **Hyperparameter Tuning**: Grid search and random search
4. **Feature Engineering**: Advanced text features
5. **Model Persistence**: Saving and loading trained models

## 1. Setup and Imports

In [ ]:
# Standard library imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn imports
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.naive_bayes import MultinomialNB, BernoulliNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC, LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    precision_recall_fscore_support, roc_auc_score, roc_curve
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder

# Text processing
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize

# Download required NLTK data
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')
    
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')
    
try:
    nltk.data.find('corpora/wordnet')
except LookupError:
    nltk.download('wordnet')

# Set up plotting
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)

print("✅ All imports successful!")

## 2. Data Preparation

Let's create a comprehensive dataset for text classification with multiple categories.

In [ ]:
# Create a multi-class text classification dataset
def create_sample_dataset():
    """Create a sample dataset with multiple text categories"""
    
    # Technology texts
    tech_texts = [
        "The new smartphone features advanced AI capabilities and 5G connectivity.",
        "Machine learning algorithms are revolutionizing data analysis.",
        "Cloud computing provides scalable infrastructure for modern applications.",
        "Blockchain technology ensures secure and transparent transactions.",
        "Virtual reality is transforming the gaming and entertainment industry.",
        "The Internet of Things connects devices for smart home automation.",
        "Cybersecurity measures protect against digital threats and attacks.",
        "Software development follows agile methodologies for better results.",
        "Data science combines statistics and programming for insights.",
        "Artificial intelligence is the future of automation and decision-making."
    ]
    
    # Sports texts
    sports_texts = [
        "The team won the championship with an outstanding performance.",
        "Football players train rigorously to improve their skills.",
        "Basketball requires teamwork and strategic planning.",
        "Tennis players compete in international tournaments worldwide.",
        "Swimming is an excellent exercise for cardiovascular health.",
        "Athletes follow strict diets to maintain peak physical condition.",
        "The Olympic Games bring together athletes from around the world.",
        "Soccer is the most popular sport globally with millions of fans.",
        "Baseball has a rich history in American sports culture.",
        "Golf requires precision and mental focus for success."
    ]
    
    # Business texts
    business_texts = [
        "The company reported strong quarterly earnings and growth.",
        "Marketing strategies focus on customer engagement and retention.",
        "Financial planning helps businesses manage resources effectively.",
        "Leadership skills are essential for successful business management.",
        "Market research provides insights into consumer behavior.",
        "Entrepreneurship requires innovation and risk-taking abilities.",
        "Supply chain management optimizes product delivery and costs.",
        "Human resources handle employee recruitment and development.",
        "Strategic planning guides long-term business objectives.",
        "Customer service excellence drives business success and loyalty."
    ]
    
    # Entertainment texts
    entertainment_texts = [
        "The movie received critical acclaim for its innovative storytelling.",
        "Music festivals bring together artists and fans from diverse backgrounds.",
        "Television shows entertain audiences with compelling narratives.",
        "Video games offer immersive experiences and interactive storytelling.",
        "Comedy shows provide laughter and entertainment for all ages.",
        "Drama series explore complex human emotions and relationships.",
        "Action movies feature spectacular stunts and thrilling sequences.",
        "Documentaries educate viewers about real-world topics and events.",
        "Reality TV shows capture authentic moments and human interactions.",
        "Animation appeals to audiences of all ages with creative storytelling."
    ]
    
    # Combine all texts and labels
    texts = tech_texts + sports_texts + business_texts + entertainment_texts
    labels = ['technology'] * len(tech_texts) + ['sports'] * len(sports_texts) + \
             ['business'] * len(business_texts) + ['entertainment'] * len(entertainment_texts)
    
    return pd.DataFrame({
        'text': texts,
        'category': labels
    })

# Create the dataset
df = create_sample_dataset()
print(f"Dataset shape: {df.shape}")
print(f"Categories: {df['category'].unique()}")
print(f"\nCategory distribution:")
print(df['category'].value_counts())

# Display sample texts
print(f"\nSample texts from each category:")
for category in df['category'].unique():
    print(f"\n{category.upper()}:")
    sample_text = df[df['category'] == category]['text'].iloc[0]
    print(f"  {sample_text}")

## 3. Text Preprocessing Pipeline

Let's create a comprehensive text preprocessing function that combines all the techniques we learned.

In [ ]:
class TextPreprocessor:
    """Advanced text preprocessing pipeline"""
    
    def __init__(self, remove_stopwords=True, use_stemming=True, use_lemmatization=False):
        self.remove_stopwords = remove_stopwords
        self.use_stemming = use_stemming
        self.use_lemmatization = use_lemmatization
        
        # Initialize NLTK components
        self.stop_words = set(stopwords.words('english'))
        self.stemmer = PorterStemmer()
        self.lemmatizer = WordNetLemmatizer()
    
    def clean_text(self, text):
        """Clean and normalize text"""
        # Convert to lowercase
        text = text.lower()
        
        # Remove special characters and numbers
        text = re.sub(r'[^a-zA-Z\\s]', '', text)
        
        # Remove extra whitespace
        text = re.sub(r'\\s+', ' ', text).strip()
        
        return text
    
    def tokenize_and_process(self, text):
        """Tokenize and apply text processing"""
        # Tokenize
        tokens = word_tokenize(text)
        
        # Remove stopwords
        if self.remove_stopwords:
            tokens = [token for token in tokens if token not in self.stop_words]
        
        # Apply stemming or lemmatization
        if self.use_stemming:
            tokens = [self.stemmer.stem(token) for token in tokens]
        elif self.use_lemmatization:
            tokens = [self.lemmatizer.lemmatize(token) for token in tokens]
        
        return ' '.join(tokens)
    
    def preprocess(self, texts):
        """Preprocess a list of texts"""
        processed_texts = []
        
        for text in texts:
            # Clean text
            cleaned_text = self.clean_text(text)
            
            # Tokenize and process
            processed_text = self.tokenize_and_process(cleaned_text)
            
            processed_texts.append(processed_text)
        
        return processed_texts

# Initialize preprocessor
preprocessor = TextPreprocessor(remove_stopwords=True, use_stemming=True)

# Preprocess the dataset
df['processed_text'] = preprocessor.preprocess(df['text'])

print("✅ Text preprocessing completed!")
print(f"\nSample original vs processed text:")
for i in range(3):
    print(f"\nOriginal: {df['text'].iloc[i]}")
    print(f"Processed: {df['processed_text'].iloc[i]}")